# 06-04. Passive Tracer Experiments in 7-box model 
# 7-box model で保存トレーサー実験を行う

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

06-03 では、7-box model の 7×7 輸送行列を作りました。  
In 06-03, we built the 7×7 transport matrix for the 7-box model.

この Notebook では、その輸送行列に保存トレーサーを流します。  
In this notebook, we run passive tracers through that transport matrix.

特に注目するのは、新しく追加された **A box** です。  
We focus especially on the newly added **A box**.

```text
N → A → D → S → P → L → N
```

N に入れた染料が、まず A に入り、その後 D へ広がるかを確認します。  
We check whether dye released in N first enters A and then spreads into D.

> **A box を追加すると、北大西洋起源の深層水が D に入る前の経路を見えるようにできる。**  
> **Adding A makes the pathway of North Atlantic-origin deep water visible before it enters D.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: 7-box で何を見たいのか / What do we want to see in 7-box?

6-box PSLNMD では、N に入れた染料は M や D へ広がりました。  
In the six-box PSLNMD model, dye released in N spread into M and D.

7-box PSLNMAD では、N で沈み込んだ水は次の経路を通ると考えます。  
In the seven-box PSLNMAD model, water sinking in N follows:

```text
N → A → D
```

したがって、染料実験で確認したいことは次です。  
Therefore, we want to check:

```text
N に入れた染料は A に先に届くか？
A と D では到達時間が違うか？
S, P, L から入れた染料は A, D にどう届くか？
```

```text
Does dye released in N reach A first?
Do A and D have different arrival times?
How do dyes released in S, P, and L reach A and D?
```

## 2. 7-box model の準備 / Prepare 7-box model

箱は次の 7 つです。  
The seven boxes are:

```text
P, S, L, N, M, A, D
```

基本経路は、

```text
N → A → D → S → P → L → N
```

です。

さらに、

```text
S ↔ M
L ↔ M
```

の交換を入れます。  
We also include exchanges:

```text
S ↔ M
L ↔ M
```

In [ ]:
boxes = ["P", "S", "L", "N", "M", "A", "D"]
surface_boxes = ["P", "S", "L", "N"]
interior_boxes = ["M", "A", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
    "A": 0.12 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())

volumes = np.array([VOLUME[b] for b in boxes])
idx = {b: i for i, b in enumerate(boxes)}

Q_DEFAULT = 2.0e7
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400

pathway = [("N", "A"), ("A", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
exchanges = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]

pd.DataFrame({
    "Box": boxes,
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
})

## 3. 輸送行列を作る / Build the transport matrix

06-03 と同じ関数を使って、輸送行列を作ります。  
We use the same functions as in 06-03 to build the transport matrix.

In [ ]:
def build_flux_matrix(pathway, Q, boxes):
    F = np.zeros((len(boxes), len(boxes)))
    local_idx = {b: i for i, b in enumerate(boxes)}
    for source, target in pathway:
        i = local_idx[target]
        j = local_idx[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges=None):
    F = build_flux_matrix(pathway, Q, boxes)
    local_idx = {b: i for i, b in enumerate(boxes)}
    if exchanges is None:
        exchanges = []
    for a, b, q in exchanges:
        ia = local_idx[a]
        ib = local_idx[b]
        F[ib, ia] += q
        F[ia, ia] -= q
        F[ia, ib] += q
        F[ib, ib] -= q
    return F

F = build_flux_matrix_with_exchange(pathway, Q_DEFAULT, boxes, exchanges)

pd.DataFrame(F, index=boxes, columns=boxes)

## 4. 保存チェック / Conservation check

保存トレーサーでは、総量が保存される必要があります。  
For a passive tracer, total amount must be conserved.

```text
total = sum(C * volume)
```

In [ ]:
def one_step_transport(c, F):
    return c + (F @ c) * DT / volumes

def total_amount(c):
    return np.sum(c * volumes)

def run_dye(source_box="N", years=1500, F=F):
    c = np.zeros(len(boxes))
    c[idx[source_box]] = 1.0
    hist = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365, "total": total_amount(c)}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_transport(c, F)

    return pd.DataFrame(hist)

test = run_dye("N", years=500, F=F)

plt.figure()
plt.plot(test["year"], test["total"])
plt.xlabel("Time [yr]")
plt.ylabel("Total dye amount")
plt.title("Conservation check")
plt.grid(True)
plt.show()

print("Relative change:", (test["total"].iloc[-1] - test["total"].iloc[0]) / test["total"].iloc[0])

### Mini exercise 1 / ミニ演習 1

総トレーサー量は保存されていますか。  
Is total tracer amount conserved?

保存されない場合、どこを確認すべきでしょうか。  
If it is not conserved, what should we check?

## 5. 染料実験 1: N に入れる / Dye experiment 1: release in N

N は北大西洋表層で、沈み込みの入口です。  
N is the North Atlantic surface box and the entrance of sinking.

PSLNMAD では、

```text
N → A → D
```

という経路を考えます。  
Therefore, dye released in N should first enter A and then D.

In [ ]:
dye_N = run_dye("N", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_N["year"], dye_N[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in N")
plt.legend()
plt.grid(True)
plt.show()

## 6. A と D への到達を比較する / Compare arrival in A and D

N に入れた染料について、A と D だけを比較します。  
For dye released in N, we compare only A and D.

In [ ]:
plt.figure()
plt.plot(dye_N["year"], dye_N["A"], label="A")
plt.plot(dye_N["year"], dye_N["D"], label="D")
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye from N: A vs D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Box": ["A", "D"],
    "Dye at 100 yr": [
        dye_N.loc[dye_N["year"] == 100, "A"].iloc[0],
        dye_N.loc[dye_N["year"] == 100, "D"].iloc[0],
    ],
    "Dye at 500 yr": [
        dye_N.loc[dye_N["year"] == 500, "A"].iloc[0],
        dye_N.loc[dye_N["year"] == 500, "D"].iloc[0],
    ],
    "Dye at 1500 yr": [
        dye_N["A"].iloc[-1],
        dye_N["D"].iloc[-1],
    ],
})

### Mini exercise 2 / ミニ演習 2

N に入れた染料は、A と D のどちらに先に現れましたか。  
Does dye released in N appear first in A or D?

これは A box の意味と整合的ですか。  
Is this consistent with the meaning of the A box?

## 7. 染料実験 2: S に入れる / Dye experiment 2: release in S

S は極域表層です。  
S is the Polar surface box.

S に入れた染料は、表層経路を通って N に戻り、その後 A, D へ入ります。  
Dye released in S follows the surface pathway to N and then enters A and D.

In [ ]:
dye_S = run_dye("S", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_S["year"], dye_S[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in S")
plt.legend()
plt.grid(True)
plt.show()

## 8. 染料実験 3: P に入れる / Dye experiment 3: release in P

P は太平洋表層です。  
P is the Pacific surface box.

P に入れた染料は、L と N を経由して A, D へ入ります。  
Dye released in P reaches A and D through L and N.

In [ ]:
dye_P = run_dye("P", years=1500, F=F)

plt.figure()
for b in boxes:
    plt.plot(dye_P["year"], dye_P[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Dye concentration")
plt.title("Dye released in P")
plt.legend()
plt.grid(True)
plt.show()

## 9. Source box による A, D 到達の違い / Source dependence of arrival in A and D

N, S, P のどこに染料を入れるかによって、A と D への到達が変わります。  
Arrival in A and D differs depending on whether dye is released in N, S, or P.

In [ ]:
plt.figure()
plt.plot(dye_N["year"], dye_N["A"], label="A: source N")
plt.plot(dye_S["year"], dye_S["A"], label="A: source S")
plt.plot(dye_P["year"], dye_P["A"], label="A: source P")
plt.xlabel("Time [yr]")
plt.ylabel("Dye in A")
plt.title("Arrival in A")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(dye_N["year"], dye_N["D"], label="D: source N")
plt.plot(dye_S["year"], dye_S["D"], label="D: source S")
plt.plot(dye_P["year"], dye_P["D"], label="D: source P")
plt.xlabel("Time [yr]")
plt.ylabel("Dye in D")
plt.title("Arrival in D")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
summary = pd.DataFrame({
    "Source": ["N", "S", "P"],
    "A at 300 yr": [
        dye_N.loc[dye_N["year"] == 300, "A"].iloc[0],
        dye_S.loc[dye_S["year"] == 300, "A"].iloc[0],
        dye_P.loc[dye_P["year"] == 300, "A"].iloc[0],
    ],
    "D at 300 yr": [
        dye_N.loc[dye_N["year"] == 300, "D"].iloc[0],
        dye_S.loc[dye_S["year"] == 300, "D"].iloc[0],
        dye_P.loc[dye_P["year"] == 300, "D"].iloc[0],
    ],
    "A at 1000 yr": [
        dye_N.loc[dye_N["year"] == 1000, "A"].iloc[0],
        dye_S.loc[dye_S["year"] == 1000, "A"].iloc[0],
        dye_P.loc[dye_P["year"] == 1000, "A"].iloc[0],
    ],
    "D at 1000 yr": [
        dye_N.loc[dye_N["year"] == 1000, "D"].iloc[0],
        dye_S.loc[dye_S["year"] == 1000, "D"].iloc[0],
        dye_P.loc[dye_P["year"] == 1000, "D"].iloc[0],
    ],
})
summary

### Mini exercise 3 / ミニ演習 3

A に最も早く届くのは、どの source box の染料ですか。  
Which source-box dye reaches A fastest?

D に最も早く届くのは、どの source box の染料ですか。  
Which source-box dye reaches D fastest?

その理由を輸送経路で説明してください。  
Explain why using the transport pathway.

## 10. 起源トレーサー / Origin tracer

起源トレーサーでは、ある表層 box を 1 に戻し、他の表層 box を 0 に戻します。  
For an origin tracer, one surface box is restored to 1 and other surface boxes are restored to 0.

ここでは N-origin, S-origin, P-origin を比較します。  
Here we compare N-origin, S-origin, and P-origin.

In [ ]:
def one_step_origin(c, F, origin_box, restoring_strength=1.0):
    new = one_step_transport(c, F)
    for b in surface_boxes:
        i = idx[b]
        target = 1.0 if b == origin_box else 0.0
        new[i] = (1.0 - restoring_strength) * new[i] + restoring_strength * target
    return new

def run_origin_tracer(origin_box="N", years=3000, F=F):
    c = np.zeros(len(boxes))
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for i, b in enumerate(boxes):
                row[b] = c[i]
            hist.append(row)
        c = one_step_origin(c, F, origin_box)
    return pd.DataFrame(hist)

origin_N = run_origin_tracer("N", years=3000, F=F)
origin_S = run_origin_tracer("S", years=3000, F=F)
origin_P = run_origin_tracer("P", years=3000, F=F)

plt.figure()
plt.plot(origin_N["year"], origin_N["A"], label="N-origin in A")
plt.plot(origin_S["year"], origin_S["A"], label="S-origin in A")
plt.plot(origin_P["year"], origin_P["A"], label="P-origin in A")
plt.xlabel("Time [yr]")
plt.ylabel("Origin tracer")
plt.title("Origin tracers in A")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(origin_N["year"], origin_N["D"], label="N-origin in D")
plt.plot(origin_S["year"], origin_S["D"], label="S-origin in D")
plt.plot(origin_P["year"], origin_P["D"], label="P-origin in D")
plt.xlabel("Time [yr]")
plt.ylabel("Origin tracer")
plt.title("Origin tracers in D")
plt.legend()
plt.grid(True)
plt.show()

## 11. A と D の起源を比較する / Compare origins in A and D

A と D で、どの表層起源の影響が大きいかを比較します。  
We compare which surface origin is important in A and D.

In [ ]:
origin_summary = pd.DataFrame({
    "Box": ["A", "D"],
    "N-origin": [origin_N["A"].iloc[-1], origin_N["D"].iloc[-1]],
    "S-origin": [origin_S["A"].iloc[-1], origin_S["D"].iloc[-1]],
    "P-origin": [origin_P["A"].iloc[-1], origin_P["D"].iloc[-1]],
})
origin_summary

In [ ]:
x = np.arange(len(origin_summary["Box"]))
width = 0.25

plt.figure()
plt.bar(x - width, origin_summary["N-origin"], width, label="N-origin")
plt.bar(x, origin_summary["S-origin"], width, label="S-origin")
plt.bar(x + width, origin_summary["P-origin"], width, label="P-origin")
plt.xticks(x, origin_summary["Box"])
plt.ylabel("Origin tracer")
plt.title("Origin tracers in A and D")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

### Mini exercise 4 / ミニ演習 4

A と D の起源トレーサーは似ていますか、違いますか。  
Are origin tracers in A and D similar or different?

A が N-origin を強く持つ理由を説明してください。  
Explain why A tends to have strong N-origin influence.

## 12. 輸送強度 Q の感度 / Sensitivity to transport strength Q

最後に、循環強度 Q を変えます。  
Finally, we change circulation strength Q.

Q が大きいほど、N 起源の染料は A と D に早く到達するはずです。  
Larger Q should make N-origin dye arrive in A and D faster.

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_Q = []

plt.figure()
for q in Q_list:
    Fq = build_flux_matrix_with_exchange(pathway, q, boxes, exchanges)
    h = run_dye("N", years=1500, F=Fq)
    plt.plot(h["year"], h["A"], label=f"A, Q={q:.1e}")
    summary_Q.append({
        "Q": q,
        "A at 300 yr": h.loc[h["year"] == 300, "A"].iloc[0],
        "D at 300 yr": h.loc[h["year"] == 300, "D"].iloc[0],
        "A at 1000 yr": h.loc[h["year"] == 1000, "A"].iloc[0],
        "D at 1000 yr": h.loc[h["year"] == 1000, "D"].iloc[0],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Dye in A")
plt.title("Transport strength and arrival in A")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_Q)

## 13. 06-04 のまとめ / Summary of 06-04

この Notebook で学んだことは次です。  
What we learned:

1. PSLNMAD の保存トレーサー実験では、A box の役割を直接見ることができる。  
   Passive tracer experiments in PSLNMAD allow us to directly see the role of A.

2. N に入れた染料は、まず A に入り、その後 D に広がる。  
   Dye released in N first enters A and then spreads into D.

3. S や P に入れた染料は、表層経路を通ってから N, A, D へ進む。  
   Dye released in S or P follows surface pathways before reaching N, A, and D.

4. Origin tracer により、A と D の水塊起源を比較できる。  
   Origin tracers allow us to compare water-mass origins in A and D.

5. Q を大きくすると、トレーサーは A と D に早く届く。  
   Larger Q makes tracers reach A and D faster.

## Key message

> **A box は、N で沈み込んだ水が古い D に入る前の通り道として可視化できる。**  
> **The A box visualizes the pathway of water sinking in N before it enters older D.**

## 14. 課題 / Exercises

### 課題 1 / Exercise 1

N に入れた染料が A に先に届く理由を説明せよ。  
Explain why dye released in N reaches A first.

### 課題 2 / Exercise 2

A と D の染料時系列の違いは何を意味するか。  
What does the difference between dye time series in A and D mean?

### 課題 3 / Exercise 3

S に入れた染料が A に届くまでの経路を説明せよ。  
Explain the pathway by which dye released in S reaches A.

### 課題 4 / Exercise 4

Origin tracer を使うことで、A と D について何が分かるか。  
What can origin tracers tell us about A and D?

### 課題 5 / Exercise 5

Q を大きくすると、A と D の染料濃度はどのように変化するか。  
How do dye concentrations in A and D change when Q is increased?

## 15. 次へ / Next step

次の Notebook では、7-box model の ideal age を扱います。  
In the next notebook, we treat ideal age in the 7-box model.

```text
06-05 Ideal Age in 7-box model
```

A と D の年齢差を調べ、ベンチレーション・O2・Δ14C への接続を考えます。  
We examine age differences between A and D and connect them to ventilation, O2, and Δ14C.